# Reproducible LoRA Fine-Tuning: Natural Language to Structured JSON

**Model:** `Qwen/Qwen2.5-0.5B-Instruct`  
**Method:** Parameter-Efficient Fine-Tuning (PEFT) with LoRA  
**Environment:** Turnkey execution for Google Colab (T4 GPU), Kaggle, or Local CUDA / CPU  

This notebook provides the complete reproducible pipeline:
1. Environment Setup & Dependency Installation
2. Dataset Generation & Zero-Leakage Validation (1,600 samples)
3. Baseline Model Evaluation on Held-Out Test Set (160 samples)
4. LoRA Adapter Configuration & Prompt-Loss Masked Training
5. Post-Fine-Tuning Evaluation & Comparative Benchmark
6. Genuine Failure Case Extraction & Error Analysis

In [ ]:
# 1. Environment Installation
!pip install -q torch transformers peft accelerate datasets pyyaml tqdm

In [ ]:
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active Device:   {torch.cuda.get_device_name(0)}")

## 2. Dataset Synthesis & Zero-Leakage Validation
Generates 1,600 schema-compliant examples partitioned strictly into:
- **Train (80%):** 1,280 samples
- **Validation (10%):** 160 samples
- **Test (10%):** 160 samples

In [ ]:
# Run dataset generator
!python scripts/generate_dataset.py

# Run dataset integrity and cross-split leakage audit
!python scripts/validate_dataset.py

## 3. Baseline Model Evaluation
Evaluates the untuned `Qwen/Qwen2.5-0.5B-Instruct` model on the 160 held-out test samples under deterministic greedy decoding.

In [ ]:
!python src/evaluate.py --test_data data/test.jsonl --output_predictions results/baseline_predictions.jsonl --output_metrics results/baseline_metrics.json

## 4. LoRA Fine-Tuning Pipeline
Trains LoRA adapters ($r=16, \alpha=32$, targeting `q_proj, k_proj, v_proj, o_proj`) using **Prompt Loss Masking** (loss computed exclusively over the target JSON output tokens).

In [ ]:
!python src/train.py

## 5. Post-Fine-Tuning Evaluation & Comparative Benchmark
Evaluates the fine-tuned checkpoint (`models/qwen2.5-0.5b-lora/best_adapter`) on the exact same 160 test samples and generates head-to-head metrics.

In [ ]:
!python src/evaluate.py --adapter_path models/qwen2.5-0.5b-lora/best_adapter --test_data data/test.jsonl --output_predictions results/finetuned_predictions.jsonl --output_metrics results/finetuned_metrics.json

# Aggregate comparison table & chart
!python scripts/compare_models.py

## 6. Failure Case Extraction & Error Analysis
Extracts genuine failure cases from actual test predictions and inspects root causes.

In [ ]:
import json
with open("results/failure_analysis.json", "r") as f:
    failures = json.load(f)

for idx, fail in enumerate(failures, 1):
    print(f"\n=== Failure Case {idx}: {fail['error_category']} ===")
    print(f"User Input:   {fail['user_input']}")
    print(f"Expected:     {fail['expected_json']}")
    print(f"Actual Model: {fail['actual_model_output']}")
    print(f"Root Cause:   {fail['explanation']}")
    print(f"Mitigation:   {fail['potential_improvement']}")